# NeoRX · Análisis de sensibilidad en VALIDACIÓN
Usa el mismo modelo ya evaluado. **No entrena, no vuelve a inferir TEST y no habilita clases.**
Agrega como entradas privadas: **neorx_avance_4**, el paquete ORIGINAL
**neorx_kaggle_entrenamiento** y **NIH Chest X-rays completo**.
También admite la carpeta `neorx-evaluation` de la evaluación terminada en lugar del avance 4.
Deja solo una salida anterior adjunta. GPU T4 e Internet activados.
Ejecuta con **Save Version → Save & Run All**.
Los escenarios de sensibilidad 80/85/90/95% son exploratorios, no criterios clínicos aprobados.
Los resultados de TEST anteriores se conservan. Una política nueva necesita evaluación independiente.

In [ ]:
# Puedes dejar ambos vacíos si adjuntas solo una salida anterior y un dataset NIH.
PREVIOUS_OUTPUT = ""  # carpeta neorx-run del avance 4, o neorx-evaluation
DATA_ROOT = ""        # carpeta que contiene el NIH completo; se detecta automáticamente

In [ ]:
from pathlib import Path
import json, os, sys, zipfile, subprocess, shutil

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)
bundles = list(INPUT.rglob('bundle_manifest.json'))
if not bundles:
    archives = list(INPUT.rglob('neorx_kaggle_entrenamiento.zip'))
    if len(archives) != 1:
        raise RuntimeError('Agrega el paquete de código como Dataset privado; debe haber una sola copia.')
    target = WORK / 'neorx-code'
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        for member in z.infolist():
            resolved = (target / member.filename).resolve()
            if not resolved.is_relative_to(target.resolve()) or (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise RuntimeError('Ruta no permitida en el ZIP')
        z.extractall(target)
    bundles = list(target.rglob('bundle_manifest.json'))
if len(bundles) != 1:
    raise RuntimeError('Hay varios paquetes de código. Deja solo la versión que vas a utilizar.')
BUNDLE = bundles[0].parent
print('Código:', BUNDLE)
print('Entradas NIH:', [str(p.parent) for p in INPUT.rglob('Data_Entry_2017.csv')])
print('Salidas anteriores:', [str(p.parent) for p in INPUT.rglob('STATUS.json')])
# PyTorch/CUDA se conservan tal como los proporciona Kaggle.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                '-r', str(BUNDLE / 'training/requirements.kaggle.txt')], check=True)
sys.path.insert(0, str(BUNDLE))
os.environ['PYTHONPATH'] = str(BUNDLE)
os.environ['PYTHONUNBUFFERED'] = '1'
from training.kaggle_run import verify_bundle
info = verify_bundle(BUNDLE)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Activa Accelerator: GPU y guarda una nueva versión con GPU.')
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)
print('Imágenes de referencia:', info['reference_images'])

In [ ]:
# Script independiente. Identidad del modelo y dataset revisados localmente.
script_path = WORK / 'analyze_validation.py'
script_path.write_text('"""Explore frozen-model operating points on VALIDATION only; never deploy them."""\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\n\n\ndef dataset_identity(manifest_text, splits):\n    rows = sorted((r[\'Image Index\'], r[\'Patient ID\'], r[\'Finding Labels\'],\n                   r.get(\'View Position\', \'\'), r[\'sha256\'])\n                  for r in csv.DictReader(io.StringIO(manifest_text)))\n    return hashlib.sha256(json.dumps([rows, splits], sort_keys=True).encode()).hexdigest()\n\n\ndef operating_points(target, scores):\n    """Exact >= thresholds, grouping tied scores; no rounded-threshold decisions."""\n    target, scores = np.asarray(target), np.asarray(scores)\n    if target.ndim != 1 or target.shape != scores.shape or len(target) == 0:\n        raise ValueError(\'Invalid prediction dimensions\')\n    if not np.isin(target, [0, 1]).all() or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():\n        raise ValueError(\'Invalid binary targets or sigmoid scores\')\n    positives, negatives = int(target.sum()), int((1 - target).sum())\n    if not positives or not negatives:\n        raise ValueError(\'Both positive and negative validation examples are required\')\n    order = np.argsort(-scores, kind=\'stable\')\n    sorted_scores, sorted_targets = scores[order], target[order]\n    ends = np.r_[np.flatnonzero(np.diff(sorted_scores) != 0), len(scores) - 1]\n    tp = np.cumsum(sorted_targets)[ends].astype(int)\n    fp = ends + 1 - tp\n    return [dict(threshold=float(sorted_scores[end]), tp=int(t), fp=int(f),\n                 fn=positives-int(t), tn=negatives-int(f),\n                 sensitivity=float(t/positives), specificity=float(1-f/negatives),\n                 precision=float(t/(t+f)), false_positives_per_1000_negatives=float(1000*f/negatives))\n            for end, t, f in zip(ends, tp, fp)]\n\n\ndef exploratory_candidates(points, targets=(0.80, 0.85, 0.90, 0.95)):\n    # Descending thresholds: first eligible point minimizes emitted positives.\n    return [dict(sensitivity_target=goal, **next(p for p in points if p[\'sensitivity\'] >= goal),\n                 enabled=False, status=\'exploratory_validation_only\') for goal in targets]\n\n\ndef write_csv(path, rows):\n    with path.open(\'w\', encoding=\'utf-8\', newline=\'\') as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef prepare_validation(bundle, previous, output, data_root, expected_checkpoint, expected_data):\n    import pandas as pd\n    import yaml\n    from training.analyze_targets import analyze\n    from training.configuration import get_target_labels\n    from training.kaggle_run import verify_bundle\n    from training.model import load_checkpoint\n    from training.validate_dataset import index_images\n\n    build = verify_bundle(bundle)\n    if json.loads((previous / \'source_bundle.json\').read_text(encoding=\'utf-8\')) != build:\n        raise ValueError(\'Use the original training code bundle\')\n    cfg = yaml.safe_load((previous / \'config.runtime.yaml\').read_text(encoding=\'utf-8\'))\n    checkpoint = previous / \'checkpoints/best.pt\'\n    if not checkpoint.exists():\n        checkpoint = previous / \'checkpoints\' / cfg[\'model\'][\'checkpoint_name\']\n    if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != expected_checkpoint:\n        raise ValueError(\'Checkpoint differs from the already reviewed frozen model\')\n    manifest_text = (previous / \'manifests/clean_manifest.csv\').read_text(encoding=\'utf-8-sig\')\n    splits = json.loads((previous / \'manifests/splits.json\').read_text(encoding=\'utf-8\'))\n    if dataset_identity(manifest_text, splits) != expected_data:\n        raise ValueError(\'Dataset identity differs: labels, patients, hashes or splits changed\')\n    analyze(manifest_text, splits)  # Metadata overlap checks, no TEST inference.\n    best = load_checkpoint(checkpoint)\n    labels = get_target_labels(cfg)\n    if best[\'pathologies\'] != labels:\n        raise ValueError(\'Checkpoint label order differs from configuration\')\n    rows = {r[\'Image Index\']: r for r in csv.DictReader(io.StringIO(manifest_text))}\n    paths = index_images(data_root)\n    val_rows = []\n    for index, name in enumerate(splits[\'val\'], 1):\n        row = dict(rows[name])\n        matches = paths.get(name, [])\n        if len(matches) != 1:\n            raise ValueError(f\'Missing or ambiguous VALIDATION image: {name}\')\n        path = matches[0]\n        if hashlib.sha256(path.read_bytes()).hexdigest() != row[\'sha256\']:\n            raise ValueError(f\'VALIDATION image changed: {name}\')\n        row[\'image_path\'] = str(path)\n        val_rows.append(row)\n        if index % 1000 == 0 or index == len(splits[\'val\']):\n            print(f\'VALIDATION verified: {index:,}/{len(splits["val"]):,}\', flush=True)\n    provenance = dict(partition=\'val\', samples=len(val_rows), checkpoint_sha256=expected_checkpoint,\n                      dataset_identity=expected_data, original_thresholds=best[\'thresholds\'],\n                      original_decision_policy=best[\'decision_policy\'],\n                      selected_epoch_zero_based=best[\'extra\'][\'epoch\'],\n                      validation_metrics_at_training=best.get(\'metrics\', {}),\n                      test_inference_performed=False, deployment_enabled=False)\n    (output / \'provenance.json\').write_text(json.dumps(provenance, indent=2), encoding=\'utf-8\')\n    return cfg, checkpoint, pd.DataFrame(val_rows), labels, best[\'thresholds\']\n\n\nclass ValidationProgress:\n    def __init__(self, loader):\n        self.loader = loader\n\n    def __iter__(self):\n        started = last = time.monotonic()\n        completed, total = 0, len(self.loader.dataset)\n        for batch in self.loader:\n            yield batch\n            completed += len(batch[2])\n            now = time.monotonic()\n            if completed == len(batch[2]) or now-last >= 30 or completed == total:\n                rate = completed / max(now-started, 1e-6)\n                print(f\'VALIDATION inference: {completed:,}/{total:,} | {rate:.1f} images/s | \'\n                      f\'ETA inference only: {(total-completed)/rate/60:.1f} min\', flush=True)\n                last = now\n\n\ndef run(args):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    import torch\n    from sklearn.metrics import average_precision_score, roc_auc_score\n    from torch.utils.data import DataLoader\n    from training.dataset import NIHDataset, build_eval_transform\n    from training.evaluate import evaluate_model, compute_metrics\n    from training.model import load_finetuned_model\n    from training.kaggle_run import discover_data_root\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'GPU required for this notebook; enable it before running\')\n    root = args.data_root or discover_data_root(Path(\'/kaggle/input\'))\n    cfg, checkpoint, frame, labels, original = prepare_validation(\n        args.bundle, args.previous, args.output, root, args.expected_checkpoint, args.expected_data)\n    model = load_finetuned_model(checkpoint, base_weights=cfg[\'model\'][\'base_weights\'],\n                                cache_dir=str(args.bundle / \'weights\')).to(\'cuda\')\n    ds = NIHDataset(frame, labels, build_eval_transform())\n    loader = DataLoader(ds, batch_size=16, num_workers=2, shuffle=False, drop_last=False, pin_memory=True)\n    y, scores, ids = evaluate_model(model, ValidationProgress(loader), torch.device(\'cuda\'))\n    if ids != frame[\'Image Index\'].tolist() or len(ids) != len(ds):\n        raise ValueError(\'Validation inference lost or reordered samples\')\n    # Store scores once; later exploration does not need another GPU run.\n    np.savez_compressed(args.output / \'validation_predictions.npz\', targets=y, probabilities=scores,\n                        image_ids=np.asarray(ids), labels=np.asarray(labels))\n    write_csv(args.output / \'validation_predictions.csv\', [dict(image_id=name, **{\n        f\'{label}_{kind}\': float(array[i, j]) for j, label in enumerate(labels)\n        for kind, array in [(\'target\', y), (\'score\', scores)]}) for i, name in enumerate(ids)])\n    candidates, baseline, discrimination = [], [], []\n    fig, axes = plt.subplots(1, 2, figsize=(12, 5))\n    for j, label in enumerate(labels):\n        points = operating_points(y[:, j], scores[:, j])\n        write_csv(args.output / f\'thresholds_{label}.csv\', points)\n        candidates.extend(dict(label=label, **p) for p in exploratory_candidates(points))\n        m = compute_metrics(y[:, j], scores[:, j], original[label])\n        tn, fp, fn, tp = np.asarray(m[\'confusion_matrix\']).ravel().tolist()\n        baseline.append(dict(label=label, threshold=original[label], sensitivity=m[\'sensitivity\'],\n                             precision=m[\'precision\'] if tp+fp else None, tp=tp, fp=fp, fn=fn, tn=tn,\n                             note=\'raw_threshold_only_not_deployed_routing\'))\n        discrimination.append(dict(label=label, auc=float(roc_auc_score(y[:, j], scores[:, j])),\n                                   average_precision=float(average_precision_score(y[:, j], scores[:, j]))))\n        axes[0].plot([p[\'sensitivity\'] for p in points], [p[\'precision\'] for p in points], label=label)\n        axes[1].plot([p[\'sensitivity\'] for p in points], [p[\'false_positives_per_1000_negatives\'] for p in points], label=label)\n    write_csv(args.output / \'comparacion_sensibilidad.csv\', candidates)\n    write_csv(args.output / \'umbrales_originales_VALIDACION.csv\', baseline)\n    write_csv(args.output / \'discriminacion_VALIDACION.csv\', discrimination)\n    for ax, title, ylabel in zip(axes, [\'VALIDACION: sensibilidad y precision\', \'VALIDACION: costo en falsas alarmas\'],\n                                  [\'Precision (valor predictivo positivo)\', \'Falsos positivos / 1000 negativos\']):\n        ax.set(xlabel=\'Sensibilidad\', ylabel=ylabel, title=title, xlim=(0, 1))\n        ax.grid(alpha=.2)\n        ax.legend(fontsize=8)\n    fig.tight_layout()\n    fig.savefig(args.output / \'sensibilidad_falsas_alarmas.png\', dpi=160)\n    plt.close(fig)\n    (args.output / \'LEEME.txt\').write_text(\n        \'Analisis exploratorio de VALIDACION, no evaluacion independiente ni autorizacion clinica.\\n\'\n        \'80/85/90/95% son escenarios para comparar, no criterios clinicos aprobados.\\n\'\n        \'Los umbrales originales y el checkpoint no se modificaron; ninguna clase se habilito.\\n\'\n        \'Una mayor sensibilidad puede implicar muchas falsas alarmas y menor precision.\\n\'\n        \'Los scores sigmoid no estan demostrados como probabilidades clinicas calibradas.\\n\'\n        \'Las categorias usan etiquetas derivadas NIH, no etiologias confirmadas.\\n\'\n        \'Cada fila es una clase binaria; no mide el enrutamiento conjunto del sistema.\\n\'\n        \'TEST ya fue consultado: una politica revisada necesita evaluacion independiente.\\n\'\n        \'Esta validacion se uso tambien para seleccionar el modelo: no demuestra generalizacion.\\n\', encoding=\'utf-8\')\n    print(\'\\nVALIDATION exploration completed. No model or deployment policy changed.\', flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    for name in (\'bundle\', \'previous\', \'output\'):\n        parser.add_argument(\'--\'+name, type=Path, required=True)\n    parser.add_argument(\'--data-root\', type=Path)\n    parser.add_argument(\'--expected-checkpoint\', required=True)\n    parser.add_argument(\'--expected-data\', required=True)\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=False)\n    status = args.output / \'STATUS.json\'\n    status.write_text(json.dumps(dict(status=\'running\', partition=\'val\')), encoding=\'utf-8\')\n    try:\n        run(args)\n    except Exception as error:\n        status.write_text(json.dumps(dict(status=\'failed\', error=str(error))), encoding=\'utf-8\')\n        raise\n    status.write_text(json.dumps(dict(status=\'validation_analysis_complete\', partition=\'val\',\n                                     deployment_enabled=False)), encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
if not PREVIOUS_OUTPUT:
    previous_candidates = [p.parent for p in INPUT.rglob('source_bundle.json')
                           if (p.parent / 'manifests/splits.json').exists()
                           and (p.parent / 'checkpoints').is_dir()]
    if len(previous_candidates) != 1:
        raise RuntimeError('Deja solo el avance 4 o la evaluación como entrada, o escribe PREVIOUS_OUTPUT. '
                           + str(previous_candidates))
    PREVIOUS_OUTPUT = str(previous_candidates[0])
print('Salida anterior:', PREVIOUS_OUTPUT)
OUTPUT = WORK / 'neorx-validation-analysis'
if OUTPUT.exists():
    raise RuntimeError('Esta salida ya existe. Guarda/descarga sus resultados; para otra ejecución usa una sesión nueva.')
args = [sys.executable, '-u', str(script_path), '--bundle', str(BUNDLE),
        '--previous', PREVIOUS_OUTPUT, '--output', str(OUTPUT),
        '--expected-checkpoint', 'ce2d1d1c31485215360fa9754139a7985fb4b5b1fdb85ecff51c460ce7fa8ac2', '--expected-data', '443b26b89c9605b80ef41d95143e7fb85930696a3db0bd381b944fa96666ecc5']
if DATA_ROOT:
    args += ['--data-root', DATA_ROOT]
log_path = WORK / 'validation_console.log'
with log_path.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(args, cwd=BUNDLE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()
if OUTPUT.exists():
    shutil.copy2(log_path, OUTPUT / 'console.log')
    # Registrar conflictos, sin modificar paquetes ajenos para ocultarlos.
    for name, command in [('pip_check.txt', ['check']), ('pip_freeze.txt', ['freeze'])]:
        result = subprocess.run([sys.executable, '-m', 'pip', *command], capture_output=True, text=True)
        (OUTPUT / name).write_text(result.stdout + result.stderr, encoding='utf-8')
    with zipfile.ZipFile(WORK / 'neorx_validacion_sensibilidad.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file():
                z.write(path, 'neorx-validation-analysis/' + path.relative_to(OUTPUT).as_posix())
if return_code:
    raise RuntimeError('Análisis incompleto: revisa validation_console.log y STATUS.json.')

In [ ]:
import pandas as pd
from IPython.display import display, FileLink, Image
print((OUTPUT / 'STATUS.json').read_text(encoding='utf-8'))
print('Resultados de VALIDACIÓN; no se habilitó ninguna clase ni se cambió el modelo.')
display(pd.read_csv(OUTPUT / 'comparacion_sensibilidad.csv'))
display(Image(filename=str(OUTPUT / 'sensibilidad_falsas_alarmas.png')))
display(FileLink('/kaggle/working/neorx_validacion_sensibilidad.zip'))